## 🥈 Complete Silver Customer Pipeline
### 01_customer_silver.ipynb

- Cell 1 → Configuration
- Cell 2 → Read Bronze
- Cell 3 → Standardize
- Cell 4 → DQ rules
- Cell 5 → Quarantine invalid records
- Cell 6 → Rank current records
- Cell 7 → Create Silver table
- Cell 8 → Final validation

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================
# Configuration
# =========================

BRONZE_TABLE = "dbx_fintech_data_platform.bronze.customers"
SILVER_TABLE = "dbx_fintech_data_platform.silver.customers"
QUARANTINE_TABLE = "dbx_fintech_data_platform.silver.customers_quarantine"

In [0]:
silver_source_df = spark.table(BRONZE_TABLE)

print("Bronze records:", silver_source_df.count())

display(silver_source_df)

In [0]:
silver_clean_df = (
    silver_source_df

    # String cleanup
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("customer_type", F.upper(F.trim(F.col("customer_type"))))

    # Data type standardization
    .withColumn("phone", F.col("phone").cast("string"))
)

display(silver_clean_df)

In [0]:
valid_countries = ["CA", "DE", "IN", "UK", "US"]

valid_customer_types = [
    "STANDARD",
    "PREMIUM",
    "VIP"
]

In [0]:
dq_condition = (
    F.col("customer_id").isNull() |
    (F.trim(F.col("customer_id")) == "") |

    F.col("email").isNull() |
    (F.trim(F.col("email")) == "") |

    F.col("phone").isNull() |
    (F.trim(F.col("phone")) == "") |

    F.col("country").isNull() |
    (F.trim(F.col("country")) == "") |
    (~F.col("country").isin(valid_countries)) |

    F.col("customer_type").isNull() |
    (F.trim(F.col("customer_type")) == "") |
    (~F.col("customer_type").isin(valid_customer_types))
)

In [0]:
invalid_df = silver_clean_df.filter(dq_condition)

valid_df = silver_clean_df.filter(~dq_condition)

In [0]:
print("Valid records:", valid_df.count())
print("Invalid records:", invalid_df.count())

In [0]:
quarantine_df = (
    invalid_df
    .withColumn(
        "dq_reason",
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "INVALID_CUSTOMER_ID"
        )
        .when(
            F.col("email").isNull() |
            (F.trim(F.col("email")) == ""),
            "INVALID_EMAIL"
        )
        .when(
            F.col("phone").isNull() |
            (F.trim(F.col("phone")) == ""),
            "INVALID_PHONE"
        )
        .when(
            F.col("country").isNull() |
            (F.trim(F.col("country")) == ""),
            "INVALID_COUNTRY"
        )
        .when(
            ~F.col("country").isin(valid_countries),
            "INVALID_COUNTRY"
        )
        .when(
            F.col("customer_type").isNull() |
            (F.trim(F.col("customer_type")) == ""),
            "INVALID_CUSTOMER_TYPE"
        )
        .when(
            ~F.col("customer_type").isin(valid_customer_types),
            "INVALID_CUSTOMER_TYPE"
        )
        .otherwise("UNKNOWN_DQ_ERROR")
    )
    .withColumn("_quarantine_timestamp", F.current_timestamp())
)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.silver
""")

In [0]:
(
    quarantine_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(QUARANTINE_TABLE)
)

In [0]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_source_date").desc()
    )
)

In [0]:
ranked_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )
)

In [0]:
silver_current_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
silver_final_df = silver_current_df.select(
    "customer_id",
    "name",
    "email",
    "phone",
    "country",
    "customer_type",
    "created_at",
    "updated_at",
    "_ingestion_timestamp",
    "_source_file",
    "_source_date"
)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.silver
""")

In [0]:
(
    silver_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
silver_count = silver_final_df.count()

print("Silver records:", silver_count)

In [0]:
unique_customers = (
    silver_final_df
    .select("customer_id")
    .distinct()
    .count()
)

print("Unique customers:", unique_customers)

In [0]:
duplicate_count = (
    silver_final_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate customer IDs:", duplicate_count)

In [0]:
silver_final_df.printSchema()